In [4]:
%pip install pandas numpy scipy scikit-learn matplotlib

  Using cached scipy-1.18.1-cp313-cp313-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl.metadata (62 kB)
  Using cached scikit_learn-1.9.1-cp313-cp313-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl.metadata (9.3 kB)
  Using cached matplotlib-3.11.2-cp313-cp313-manylinux2014_x86_64.manylinux_2_17_x86_64.whl.metadata (80 kB)
  Using cached joblib-1.6.0-py3-none-any.whl.metadata (6.5 kB)
  Using cached contourpy-1.4.0-cp313-cp313-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl.metadata (3.8 kB)
  Using cached cycler-0.12.1-py3-none-any.whl.metadata (3.8 kB)
  Using cached kiwisolver-1.5.1-cp313-cp313-manylinux2014_x86_64.manylinux_2_17_x86_64.whl.metadata (5.2 kB)
  Using cached cloudpickle-3.1.2-py3-none-any.whl.metadata (7.1 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 82.2 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.7/16.7 MB 95.9 MB/s  0:00:00
Using cached scipy-1.18.1-cp313-cp313-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl (35.3 MB)
Using cached 

In [5]:
import pandas as pd
import numpy as np

# 1. Load data
df = pd.read_csv(
    "45stocks_v3.csv",
    parse_dates=["DlyCalDt"]
)

# 2. Sort by stock and date
df = df.sort_values(
    ["PERMNO", "DlyCalDt"]
).reset_index(drop=True)

# 3. Verify original columns
original_columns = [
    "PERMNO", "Ticker", "DlyCalDt",
    "DlyOpen", "DlyHigh", "DlyLow",
    "DlyClose", "DlyVol",
    "DlyCap", "sprtrn"
]

assert df.columns.tolist() == original_columns
assert df["Ticker"].nunique() == 45
assert len(df) == 124470
assert not df.duplicated(["PERMNO", "DlyCalDt"]).any()
assert not df.isna().any().any()

# 4. Verify OHLCV
assert (df[["DlyOpen", "DlyHigh", "DlyLow",
            "DlyClose", "DlyVol"]] > 0).all().all()

assert (df["DlyHigh"] >= df[["DlyOpen", "DlyClose", "DlyLow"]].max(axis=1)).all()
assert (df["DlyLow"] <= df[["DlyOpen", "DlyClose", "DlyHigh"]].min(axis=1)).all()

# 5. Summary
print("Data loaded successfully.")
print(f"Shape: {df.shape}")
print(f"Stocks: {df['Ticker'].nunique()}")
print(f"Date range: {df['DlyCalDt'].min().date()} to {df['DlyCalDt'].max().date()}")
print(f"Missing values: {df.isna().sum().sum()}")
print(f"Duplicate stock-date rows: {df.duplicated(['PERMNO', 'DlyCalDt']).sum()}")
print(f"Columns: {df.columns.tolist()}")

Data loaded successfully.
Shape: (124470, 10)
Stocks: 45
Date range: 2015-01-02 to 2025-12-31
Missing values: 0
Duplicate stock-date rows: 0
Columns: ['PERMNO', 'Ticker', 'DlyCalDt', 'DlyOpen', 'DlyHigh', 'DlyLow', 'DlyClose', 'DlyVol', 'DlyCap', 'sprtrn']


In [6]:
# ==================================================
# Cell 2: Returns & Reversal (7 Features)
# ==================================================

# 1. Historical returns
g = df.groupby("PERMNO")["DlyClose"]

for h in [1, 3, 5, 10, 20, 39]:
    df[f"RET_{h}"] = g.pct_change(
        periods=h,
        fill_method=None
    )

# 2. Positive return frequency (20 days)
up = (df["RET_1"] > 0).astype(float)
up = up.mask(df["RET_1"].isna())

df["UP_RATIO_20"] = (
    up.groupby(df["PERMNO"])
      .transform(lambda s: s.rolling(
          window=20,
          min_periods=20
      ).mean())
)

# 3. Validation
features_A = [
    "RET_1",
    "RET_3",
    "RET_5",
    "RET_10",
    "RET_20",
    "RET_39",
    "UP_RATIO_20"
]

assert len(features_A) == 7
assert (df["UP_RATIO_20"].dropna().between(0, 1)).all()
assert np.isfinite(df[features_A].dropna().to_numpy()).all()

# 4. Summary
print("Returns & Reversal completed.")
print(f"Features created: {len(features_A)}")
print(f"Current shape: {df.shape}")
print("\nNon-missing observations:")
print(df[features_A].count())

display(df[["Ticker", "DlyCalDt"] + features_A].tail())

Returns & Reversal completed.
Features created: 7
Current shape: (124470, 17)

Non-missing observations:
RET_1          124425
RET_3          124335
RET_5          124245
RET_10         124020
RET_20         123570
RET_39         122715
UP_RATIO_20    123570
dtype: int64


,Ticker,DlyCalDt,RET_1,RET_3,RET_5,RET_10,RET_20,RET_39,UP_RATIO_20
124465,AVGO,2025-12-24,0.002576,0.028969,0.074229,-0.151948,-0.090409,-0.092647,0.65
124466,AVGO,2025-12-26,0.005454,0.031278,0.067449,-0.133474,-0.114294,-0.064653,0.65
124467,AVGO,2025-12-29,-0.007781,0.000200,0.026531,-0.029283,-0.132941,-0.054757,0.60
124468,AVGO,2025-12-30,0.001317,-0.001056,0.024601,0.029546,-0.093841,-0.035030,0.65
124469,AVGO,2025-12-31,-0.010719,-0.017124,-0.009218,0.014064,-0.092958,-0.016594,0.65


In [7]:
# ==================================================
# Cell 3: Trend & Path Quality (6 Features)
# ==================================================

# 1. Group by stock
g = df.groupby("PERMNO")
close = df["DlyClose"]
log_close = np.log(close)

# 2. Moving average deviations
for h in [10, 20]:
    ma = g["DlyClose"].transform(
        lambda s: s.rolling(h, min_periods=h).mean()
    )
    df[f"MA_DEV_{h}"] = close / ma - 1

# 3. Breakout position (20 days)
high_20 = g["DlyHigh"].transform(
    lambda s: s.rolling(20, min_periods=20).max()
)

low_20 = g["DlyLow"].transform(
    lambda s: s.rolling(20, min_periods=20).min()
)

price_range = (high_20 - low_20).replace(0, np.nan)

df["BREAKOUT_POS_20"] = (
    (close - low_20) / price_range
)

# 4. Rolling trend regression (log price vs. time)
def trend_regression(y, return_t=False):
    n = len(y)
    x = np.arange(n, dtype=float)
    x = x - x.mean()
    yc = y - y.mean()

    sxx = np.dot(x, x)
    sxy = np.dot(x, yc)
    syy = np.dot(yc, yc)

    slope = sxy / sxx

    if not return_t:
        return slope

    sse = max(syy - sxy**2 / sxx, 0.0)

    if sse <= 0:
        return np.nan

    se = np.sqrt((sse / (n - 2)) / sxx)
    return slope / se

df["TREND_SLOPE_20"] = (
    log_close.groupby(df["PERMNO"]).transform(
        lambda s: s.rolling(20, min_periods=20).apply(
            trend_regression, raw=True
        )
    )
)

df["TREND_T_40"] = (
    log_close.groupby(df["PERMNO"]).transform(
        lambda s: s.rolling(40, min_periods=40).apply(
            lambda y: trend_regression(y, return_t=True),
            raw=True
        )
    )
)

# 5. Trend efficiency ratio (20 returns, 21 prices)
log_ret = log_close.groupby(df["PERMNO"]).diff()

net_move = (
    log_close -
    log_close.groupby(df["PERMNO"]).shift(20)
).abs()

total_move = (
    log_ret.abs()
    .groupby(df["PERMNO"])
    .transform(
        lambda s: s.rolling(20, min_periods=20).sum()
    )
    .replace(0, np.nan)
)

df["EFF_RATIO_20"] = net_move / total_move

# 6. Validation
features_B = [
    "MA_DEV_10",
    "MA_DEV_20",
    "BREAKOUT_POS_20",
    "TREND_SLOPE_20",
    "TREND_T_40",
    "EFF_RATIO_20"
]

assert len(features_B) == 6
assert df.shape == (124470, 23)
assert np.isfinite(df[features_B].dropna().to_numpy()).all()

assert df["BREAKOUT_POS_20"].dropna().between(
    -1e-10, 1 + 1e-10
).all()

assert df["EFF_RATIO_20"].dropna().between(
    -1e-10, 1 + 1e-10
).all()

# 7. Summary
print("Trend & Path Quality completed.")
print(f"Features created: {len(features_B)}")
print(f"Total features: {len(features_A) + len(features_B)}")
print(f"Current shape: {df.shape}")

print("\nNon-missing observations:")
print(df[features_B].count())

display(df[["Ticker", "DlyCalDt"] + features_B].tail())

Trend & Path Quality completed.
Features created: 6
Total features: 13
Current shape: (124470, 23)

Non-missing observations:
MA_DEV_10          124065
MA_DEV_20          123615
BREAKOUT_POS_20    123615
TREND_SLOPE_20     123615
TREND_T_40         122715
EFF_RATIO_20       123570
dtype: int64


,Ticker,DlyCalDt,MA_DEV_10,MA_DEV_20,BREAKOUT_POS_20,TREND_SLOPE_20,TREND_T_40,EFF_RATIO_20
124465,AVGO,2025-12-24,0.005033,-0.056657,0.309046,-0.009992,-0.161925,0.189161
124466,AVGO,2025-12-26,0.026492,-0.045672,0.329542,-0.009577,-0.035526,0.255874
124467,AVGO,2025-12-29,0.021644,-0.046173,0.300139,-0.008879,-0.028132,0.304361
124468,AVGO,2025-12-30,0.019995,-0.040171,0.305076,-0.008622,-0.078570,0.230662
124469,AVGO,2025-12-31,0.007651,-0.045817,0.264835,-0.008557,-0.243723,0.228905


In [8]:
# ==================================================
# Cell 4: Intraday & Overnight (4 Features)
# ==================================================

# 1. Previous close by stock
prev_close = df.groupby("PERMNO")["DlyClose"].shift(1)

# 2. Overnight gap
df["GAP_1"] = df["DlyOpen"] / prev_close - 1

# 3. Cumulative overnight return (5 days)
overnight_log = np.log1p(df["GAP_1"])

rolling_overnight = (
    overnight_log.groupby(df["PERMNO"])
    .transform(
        lambda s: s.rolling(5, min_periods=5).sum()
    )
)

df["OVERNIGHT_RET_5"] = np.expm1(rolling_overnight)

# 4. Close Location Value (-1 to 1)
daily_range = df["DlyHigh"] - df["DlyLow"]

df["CLV_1"] = (
    (2 * df["DlyClose"] - df["DlyHigh"] - df["DlyLow"])
    / daily_range.replace(0, np.nan)
)

# 5. Daily price range
df["RANGE_PCT_1"] = daily_range / prev_close

# 6. Validation
features_C = [
    "GAP_1",
    "OVERNIGHT_RET_5",
    "CLV_1",
    "RANGE_PCT_1"
]

assert len(features_C) == 4
assert df.shape == (124470, 27)

assert df["CLV_1"].dropna().between(
    -1 - 1e-10, 1 + 1e-10
).all()

assert (df["RANGE_PCT_1"].dropna() >= 0).all()

values = df[features_C].to_numpy(dtype=float)
assert np.isfinite(values[~np.isnan(values)]).all()

# 7. Summary
print("Intraday & Overnight completed.")
print(f"Features created: {len(features_C)}")
print(f"Total features: {len(features_A) + len(features_B) + len(features_C)}")
print(f"Current shape: {df.shape}")

print("\nNon-missing observations:")
print(df[features_C].count())

display(df[["Ticker", "DlyCalDt"] + features_C].tail())

Intraday & Overnight completed.
Features created: 4
Total features: 17
Current shape: (124470, 27)

Non-missing observations:
GAP_1              124425
OVERNIGHT_RET_5    124245
CLV_1              124470
RANGE_PCT_1        124425
dtype: int64


,Ticker,DlyCalDt,GAP_1,OVERNIGHT_RET_5,CLV_1,RANGE_PCT_1
124465,AVGO,2025-12-24,0.003908,0.060899,0.076923,0.016375
124466,AVGO,2025-12-26,0.002398,0.031857,0.592727,0.015704
124467,AVGO,2025-12-29,-0.009684,0.011273,0.666667,0.016017
124468,AVGO,2025-12-30,0.001660,-0.003646,-0.674556,0.009674
124469,AVGO,2025-12-31,-0.000343,-0.002118,-0.695067,0.012748


In [9]:
# ==================================================
# Cell 5: Volatility & Risk (7 Features)
# ==================================================

# 1. Basic variables
stock = df["PERMNO"]
close = df["DlyClose"]
high = df["DlyHigh"]
low = df["DlyLow"]
ret = df["RET_1"]

prev_close = close.groupby(stock).shift(1)

# 2. Realized volatility (5 / 20 days)
for h in [5, 20]:
    df[f"RV_{h}"] = (
        ret.groupby(stock)
        .transform(
            lambda s: s.rolling(
                h, min_periods=h
            ).std(ddof=1)
        )
    )

# 3. Downside RMS volatility (20 days)
downside_sq = np.minimum(ret, 0) ** 2

df["DOWNSIDE_RMS_20"] = np.sqrt(
    downside_sq.groupby(stock)
    .transform(
        lambda s: s.rolling(
            20, min_periods=20
        ).mean()
    )
)

# 4. Average True Range / Price (14 days)
true_range = pd.concat(
    [
        high - low,
        (high - prev_close).abs(),
        (low - prev_close).abs()
    ],
    axis=1
).max(axis=1)

# First day of each stock has no previous close
true_range = true_range.mask(prev_close.isna())

atr_14 = (
    true_range.groupby(stock)
    .transform(
        lambda s: s.rolling(
            14, min_periods=14
        ).mean()
    )
)

df["ATR_PCT_14"] = atr_14 / close

# 5. Parkinson volatility (20 days)
hl_sq = np.log(high / low) ** 2

parkinson_mean = (
    hl_sq.groupby(stock)
    .transform(
        lambda s: s.rolling(
            20, min_periods=20
        ).mean()
    )
)

df["PARKINSON_VOL_20"] = np.sqrt(
    parkinson_mean / (4 * np.log(2))
)

# 6. Overnight volatility (20 days)
overnight_log = np.log1p(df["GAP_1"])

df["OVERNIGHT_VOL_20"] = (
    overnight_log.groupby(stock)
    .transform(
        lambda s: s.rolling(
            20, min_periods=20
        ).std(ddof=1)
    )
)

# 7. Maximum drawdown (20 closing prices)
def rolling_max_drawdown(prices):
    running_max = np.maximum.accumulate(prices)
    drawdowns = 1 - prices / running_max
    return np.max(drawdowns)

df["MAX_DRAWDOWN_20"] = (
    close.groupby(stock)
    .transform(
        lambda s: s.rolling(
            20, min_periods=20
        ).apply(
            rolling_max_drawdown,
            raw=True
        )
    )
)

# 8. Validation
features_D = [
    "RV_5",
    "RV_20",
    "DOWNSIDE_RMS_20",
    "ATR_PCT_14",
    "PARKINSON_VOL_20",
    "OVERNIGHT_VOL_20",
    "MAX_DRAWDOWN_20"
]

assert len(features_D) == 7
assert df.shape == (124470, 34)

values = df[features_D].to_numpy(dtype=float)

assert np.isfinite(values[~np.isnan(values)]).all()
assert (df[features_D].dropna() >= 0).all().all()
assert df["MAX_DRAWDOWN_20"].dropna().between(0, 1).all()

# 9. Summary
total_features = sum(
    len(x) for x in [features_A, features_B, features_C, features_D]
)

print("Volatility & Risk completed.")
print(f"Features created: {len(features_D)}")
print(f"Total features: {total_features}")
print(f"Current shape: {df.shape}")

print("\nNon-missing observations:")
print(df[features_D].count())

display(df[["Ticker", "DlyCalDt"] + features_D].tail())

Volatility & Risk completed.
Features created: 7
Total features: 24
Current shape: (124470, 34)

Non-missing observations:
RV_5                124245
RV_20               123570
DOWNSIDE_RMS_20     123570
ATR_PCT_14          123840
PARKINSON_VOL_20    123615
OVERNIGHT_VOL_20    123570
MAX_DRAWDOWN_20     123615
dtype: int64


,Ticker,DlyCalDt,RV_5,RV_20,DOWNSIDE_RMS_20,ATR_PCT_14,PARKINSON_VOL_20,OVERNIGHT_VOL_20,MAX_DRAWDOWN_20
124465,AVGO,2025-12-24,0.012730,0.036105,0.031892,0.049032,0.023926,0.020989,0.210548
124466,AVGO,2025-12-26,0.013366,0.035153,0.031892,0.047347,0.023333,0.020994,0.210548
124467,AVGO,2025-12-29,0.011158,0.034869,0.031939,0.045754,0.023352,0.021072,0.210548
124468,AVGO,2025-12-30,0.011278,0.033886,0.030535,0.044053,0.023117,0.020567,0.210548
124469,AVGO,2025-12-31,0.007014,0.033876,0.030517,0.042344,0.022687,0.020527,0.210548


In [10]:
# ==================================================
# Cell 6: Volume & Liquidity (6 Features)
# ==================================================

# 1. Basic variables
stock = df["PERMNO"]
close = df["DlyClose"]
volume = df["DlyVol"]
ret = df["RET_1"]

# Estimated dollar volume
dvol = close * volume

# CRSP DlyCap is in thousands of USD
mcap_usd = df["DlyCap"] * 1000

assert (dvol > 0).all()
assert (mcap_usd > 0).all()

# 2. Relative dollar volume (5 / 20 days)
dvol_5 = (
    dvol.groupby(stock)
    .transform(
        lambda s: s.rolling(5, min_periods=5).mean()
    )
)

dvol_20 = (
    dvol.groupby(stock)
    .transform(
        lambda s: s.rolling(20, min_periods=20).mean()
    )
)

df["REL_DVOL_5_20"] = dvol_5 / dvol_20 - 1

# 3. Log average dollar volume (20 days)
df["LOG_DVOL_20"] = np.log(dvol_20)

# 4. Average daily turnover (20 days)
daily_turnover = dvol / mcap_usd

df["TURNOVER_20"] = (
    daily_turnover.groupby(stock)
    .transform(
        lambda s: s.rolling(20, min_periods=20).mean()
    )
)

# 5. Amihud illiquidity (20 returns)
daily_illiq = ret.abs() / dvol

illiq_20 = (
    daily_illiq.groupby(stock)
    .transform(
        lambda s: s.rolling(20, min_periods=20).mean()
    )
)

df["LOG_ILLIQ_20"] = np.log(
    illiq_20.where(illiq_20 > 0)
)

# 6. Return–dollar volume correlation (20 days)
log_dvol = np.log(dvol)
dlog_dvol = log_dvol.groupby(stock).diff()

df["RET_DVOL_CORR_20"] = (
    ret.groupby(stock)
    .transform(
        lambda s: s.rolling(
            20, min_periods=20
        ).corr(dlog_dvol.loc[s.index])
    )
)

# 7. Signed dollar volume balance (20 days)
signed_dvol = np.sign(ret) * dvol

signed_sum_20 = (
    signed_dvol.groupby(stock)
    .transform(
        lambda s: s.rolling(20, min_periods=20).sum()
    )
)

dvol_sum_20 = (
    dvol.groupby(stock)
    .transform(
        lambda s: s.rolling(20, min_periods=20).sum()
    )
)

df["SIGNED_DVOL_BAL_20"] = (
    signed_sum_20 / dvol_sum_20
)

# 8. Validation
features_E = [
    "REL_DVOL_5_20",
    "LOG_DVOL_20",
    "TURNOVER_20",
    "LOG_ILLIQ_20",
    "RET_DVOL_CORR_20",
    "SIGNED_DVOL_BAL_20"
]

assert len(features_E) == 6
assert df.shape == (124470, 40)

values = df[features_E].to_numpy(dtype=float)
assert np.isfinite(values[~np.isnan(values)]).all()

assert (df["TURNOVER_20"].dropna() > 0).all()

assert df["RET_DVOL_CORR_20"].dropna().between(
    -1 - 1e-10, 1 + 1e-10
).all()

assert df["SIGNED_DVOL_BAL_20"].dropna().between(
    -1 - 1e-10, 1 + 1e-10
).all()

# 9. Summary
total_features = sum(
    len(x) for x in [
        features_A, features_B, features_C,
        features_D, features_E
    ]
)

print("Volume & Liquidity completed.")
print(f"Features created: {len(features_E)}")
print(f"Total features: {total_features}")
print(f"Current shape: {df.shape}")

print("\nNon-missing observations:")
print(df[features_E].count())

display(df[["Ticker", "DlyCalDt"] + features_E].tail())

Volume & Liquidity completed.
Features created: 6
Total features: 30
Current shape: (124470, 40)

Non-missing observations:
REL_DVOL_5_20         123615
LOG_DVOL_20           123615
TURNOVER_20           123615
LOG_ILLIQ_20          123570
RET_DVOL_CORR_20      123570
SIGNED_DVOL_BAL_20    123570
dtype: int64


,Ticker,DlyCalDt,REL_DVOL_5_20,LOG_DVOL_20,TURNOVER_20,LOG_ILLIQ_20,RET_DVOL_CORR_20,SIGNED_DVOL_BAL_20
124465,AVGO,2025-12-24,0.246836,23.444264,0.008908,-27.128277,-0.136210,0.191661
124466,AVGO,2025-12-26,0.104264,23.424289,0.008766,-27.186027,-0.124534,0.175351
124467,AVGO,2025-12-29,-0.519944,23.431743,0.008855,-27.236052,-0.094388,0.131272
124468,AVGO,2025-12-30,-0.567714,23.421435,0.008787,-27.401007,-0.053908,0.181733
124469,AVGO,2025-12-31,-0.623568,23.410921,0.008716,-27.375485,-0.053346,0.194224


In [11]:
# ==================================================
# Cell 7: Market & Systematic Risk (5 Features)
# ==================================================

# 1. Basic variables
stock = df["PERMNO"]
ret = df["RET_1"]
market = df["sprtrn"]

assert (market > -1).all()

# 2. S&P 500 cumulative return (5 days)
market_log = np.log1p(market)

df["SP_RET_5"] = np.expm1(
    market_log.groupby(stock)
    .transform(
        lambda s: s.rolling(5, min_periods=5).sum()
    )
)

# 3. S&P 500 volatility (20 days)
df["SP_VOL_20"] = (
    market.groupby(stock)
    .transform(
        lambda s: s.rolling(
            20, min_periods=20
        ).std(ddof=1)
    )
)

# 4. Stock return relative to S&P 500 (20 days)
sp_ret_20 = np.expm1(
    market_log.groupby(stock)
    .transform(
        lambda s: s.rolling(20, min_periods=20).sum()
    )
)

df["REL_RET_20"] = df["RET_20"] - sp_ret_20

# 5. Rolling market regression (39 daily returns)
window = 39

market_var = (
    market.groupby(stock)
    .transform(
        lambda s: s.rolling(
            window, min_periods=window
        ).var(ddof=1)
    )
)

stock_var = (
    ret.groupby(stock)
    .transform(
        lambda s: s.rolling(
            window, min_periods=window
        ).var(ddof=1)
    )
)

market_cov = (
    ret.groupby(stock)
    .transform(
        lambda s: s.rolling(
            window, min_periods=window
        ).cov(market.loc[s.index])
    )
)

# Avoid division by zero
market_var = market_var.where(market_var > 0)

# 6. Rolling Beta
df["BETA_39"] = market_cov / market_var

# 7. Idiosyncratic volatility (OLS residual std)
residual_var = (
    (window - 1) / (window - 2)
    * (stock_var - market_cov**2 / market_var)
)

df["IDIO_VOL_39"] = np.sqrt(
    residual_var.clip(lower=0)
)

# 8. Validation
features_F = [
    "SP_RET_5",
    "SP_VOL_20",
    "REL_RET_20",
    "BETA_39",
    "IDIO_VOL_39"
]

assert len(features_F) == 5
assert df.shape == (124470, 45)

values = df[features_F].to_numpy(dtype=float)
assert np.isfinite(values[~np.isnan(values)]).all()

assert (df["SP_VOL_20"].dropna() >= 0).all()
assert (df["IDIO_VOL_39"].dropna() >= 0).all()

# 9. Summary
total_features = sum(
    len(x) for x in [
        features_A, features_B, features_C,
        features_D, features_E, features_F
    ]
)

print("Market & Systematic Risk completed.")
print(f"Features created: {len(features_F)}")
print(f"Total features: {total_features}")
print(f"Current shape: {df.shape}")

print("\nNon-missing observations:")
print(df[features_F].count())

display(df[["Ticker", "DlyCalDt"] + features_F].tail())

Market & Systematic Risk completed.
Features created: 5
Total features: 35
Current shape: (124470, 45)

Non-missing observations:
SP_RET_5       124290
SP_VOL_20      123615
REL_RET_20     123570
BETA_39        122715
IDIO_VOL_39    122715
dtype: int64


,Ticker,DlyCalDt,SP_RET_5,SP_VOL_20,REL_RET_20,BETA_39,IDIO_VOL_39
124465,AVGO,2025-12-24,0.031334,0.005743,-0.114968,2.855493,0.026832
124466,AVGO,2025-12-26,0.022905,0.005592,-0.131516,2.880178,0.026851
124467,AVGO,2025-12-29,0.010423,0.005568,-0.141212,2.896448,0.026576
124468,AVGO,2025-12-30,0.002580,0.005422,-0.106112,2.906671,0.026331
124469,AVGO,2025-12-31,-0.009305,0.005685,-0.095318,2.897859,0.026382


In [12]:
# ==================================================
# Cell 8: Peer & Cross-sectional (4 Features)
# ==================================================

# 1. Define research groups (5 groups × 9 stocks)
research_groups = {
    "Technology": [
        "AAPL", "MSFT", "NVDA", "AMD", "ADBE",
        "AVGO", "ORCL", "CRM", "AMAT"
    ],
    "Consumer": [
        "AMZN", "MCD", "SBUX", "NKE", "COST",
        "WMT", "HD", "KO", "TJX"
    ],
    "Energy": [
        "XOM", "COP", "EOG", "SLB", "VLO",
        "WMB", "LNG", "OXY", "PSX"
    ],
    "Utilities": [
        "NEE", "DUK", "SO", "AEP", "EIX",
        "AWK", "D", "SRE", "XEL"
    ],
    "Financial": [
        "JPM", "BLK", "PGR", "CME", "V",
        "BAC", "MS", "SCHW", "AXP"
    ]
}

# 2. Map stocks to research groups
ticker_to_group = {
    ticker: group
    for group, tickers in research_groups.items()
    for ticker in tickers
}

assert len(ticker_to_group) == 45
assert set(df["Ticker"]) == set(ticker_to_group)

research_group = df["Ticker"].map(ticker_to_group)

# Group by date and research group
keys = [df["DlyCalDt"], research_group]

group_size = df.groupby(keys)["Ticker"].transform("size")
assert (group_size == 9).all()

# Verify complete return observations within each group-date
for col in ["RET_1", "RET_5", "RET_20"]:
    valid_count = df.groupby(keys)[col].transform("count")
    assert valid_count.isin([0, 9]).all()

# 3. Peer cumulative return (5 days, excluding self)
peer_ret_5_sum = (
    df.groupby(keys)["RET_5"].transform("sum")
)

df["PEER_RET_5"] = (
    peer_ret_5_sum - df["RET_5"]
) / 8

# 4. Peer-relative return (20 days, excluding self)
peer_ret_20_sum = (
    df.groupby(keys)["RET_20"].transform("sum")
)

peer_ret_20 = (
    peer_ret_20_sum - df["RET_20"]
) / 8

df["PEER_REL_RET_20"] = (
    df["RET_20"] - peer_ret_20
)

# 5. Peer portfolio volatility (20 days)
# Daily equal-weight return of the other 8 stocks
peer_daily_sum = (
    df.groupby(keys)["RET_1"].transform("sum")
)

peer_daily_ret = (
    peer_daily_sum - df["RET_1"]
) / 8

df["PEER_VOL_20"] = (
    peer_daily_ret.groupby(df["PERMNO"])
    .transform(
        lambda s: s.rolling(
            20, min_periods=20
        ).std(ddof=1)
    )
)

# 6. Cross-sectional return rank (45 stocks)
df["XS_RANK_20"] = (
    df.groupby("DlyCalDt")["RET_20"]
    .rank(method="average", pct=True)
)

# 7. Validation
features_G = [
    "PEER_RET_5",
    "PEER_REL_RET_20",
    "PEER_VOL_20",
    "XS_RANK_20"
]

assert len(features_G) == 4
assert df.shape == (124470, 49)

values = df[features_G].to_numpy(dtype=float)
assert np.isfinite(values[~np.isnan(values)]).all()

assert (df["PEER_VOL_20"].dropna() >= 0).all()

assert df["XS_RANK_20"].dropna().between(0, 1).all()

expected_counts = {
    "PEER_RET_5": 124245,
    "PEER_REL_RET_20": 123570,
    "PEER_VOL_20": 123570,
    "XS_RANK_20": 123570
}

for col, expected in expected_counts.items():
    assert df[col].count() == expected, col

# 8. Summary
total_features = sum(
    len(x) for x in [
        features_A, features_B, features_C,
        features_D, features_E, features_F,
        features_G
    ]
)

print("Peer & Cross-sectional completed.")
print(f"Features created: {len(features_G)}")
print(f"Total features: {total_features}")
print(f"Current shape: {df.shape}")
print(f"Research groups: {len(research_groups)}")
print("Peer method: Leave-One-Out (8 stocks)")

print("\nNon-missing observations:")
print(df[features_G].count())

display(df[["Ticker", "DlyCalDt"] + features_G].tail())

Peer & Cross-sectional completed.
Features created: 4
Total features: 39
Current shape: (124470, 49)
Research groups: 5
Peer method: Leave-One-Out (8 stocks)

Non-missing observations:
PEER_RET_5         124245
PEER_REL_RET_20    123570
PEER_VOL_20        123570
XS_RANK_20         123570
dtype: int64


,Ticker,DlyCalDt,PEER_RET_5,PEER_REL_RET_20,PEER_VOL_20,XS_RANK_20
124465,AVGO,2025-12-24,0.050117,-0.144301,0.011557,0.022222
124466,AVGO,2025-12-26,0.041840,-0.157332,0.011250,0.022222
124467,AVGO,2025-12-29,0.014163,-0.169766,0.011292,0.022222
124468,AVGO,2025-12-30,0.003060,-0.121876,0.011255,0.022222
124469,AVGO,2025-12-31,-0.005271,-0.106748,0.011338,0.022222


In [13]:
# ==================================================
# Cell 9: Size (1 Feature)
# ==================================================

# 1. Market capitalization in USD
mcap_usd = df["DlyCap"] * 1000

assert (mcap_usd > 0).all()

# 2. Log market capitalization
df["LOG_MCAP"] = np.log(mcap_usd)

# 3. Validation
features_H = ["LOG_MCAP"]

feature_groups = [
    features_A, features_B, features_C, features_D,
    features_E, features_F, features_G, features_H
]

all_features = [
    feature for group in feature_groups
    for feature in group
]

assert len(features_H) == 1
assert len(all_features) == 40
assert len(set(all_features)) == 40
assert df.shape == (124470, 50)

assert df["LOG_MCAP"].notna().all()
assert np.isfinite(df["LOG_MCAP"]).all()

# 4. Summary
print("Size completed.")
print(f"Features created: {len(features_H)}")
print(f"Total features: {len(all_features)}")
print(f"Current shape: {df.shape}")

print("\nNon-missing observations:")
print(df[features_H].count())

print("\nFeature group summary:")
for i, group in enumerate(feature_groups, start=1):
    print(f"Group {i}: {len(group)} features")

display(
    df[["Ticker", "DlyCalDt", "DlyCap", "LOG_MCAP"]].tail()
)

Size completed.
Features created: 1
Total features: 40
Current shape: (124470, 50)

Non-missing observations:
LOG_MCAP    124470
dtype: int64

Feature group summary:
Group 1: 7 features
Group 2: 6 features
Group 3: 4 features
Group 4: 7 features
Group 5: 6 features
Group 6: 5 features
Group 7: 4 features
Group 8: 1 features


,Ticker,DlyCalDt,DlyCap,LOG_MCAP
124465,AVGO,2025-12-24,1.660393e+09,28.138075
124466,AVGO,2025-12-26,1.669448e+09,28.143514
124467,AVGO,2025-12-29,1.656458e+09,28.135703
124468,AVGO,2025-12-30,1.658639e+09,28.137018
124469,AVGO,2025-12-31,1.640860e+09,28.126242


In [14]:
# ==================================================
# Cell 10: Final Validation & CSV Export
# ==================================================

from pathlib import Path

# 1. Basic structure validation
assert df.shape == (124470, 50)
assert df["PERMNO"].nunique() == 45
assert len(all_features) == 40
assert len(set(all_features)) == 40
assert df.columns.tolist() == original_columns + all_features

assert not df.duplicated(["PERMNO", "DlyCalDt"]).any()
assert (df.groupby("PERMNO").size() == 2766).all()
assert df[original_columns].notna().all().all()

# 2. Check NaN / Inf in all features
feature_data = df[all_features]

nan_counts = feature_data.isna().sum()
inf_counts = np.isinf(feature_data.to_numpy(dtype=float)).sum()

assert inf_counts == 0

# 3. Validate 40-day warm-up period
day_index = df.groupby("PERMNO").cumcount()

warmup_mask = day_index < 39
valid_mask = feature_data.notna().all(axis=1)

# All missing features must be confined to warm-up periods
assert feature_data.loc[~warmup_mask].notna().all().all()

# All observations after warm-up must be complete
assert (valid_mask == ~warmup_mask).all()

total_rows = len(df)
warmup_rows = warmup_mask.sum()
valid_rows = valid_mask.sum()

assert warmup_rows == 45 * 39
assert valid_rows == 122715

# 4. Verify original 10 columns remain unchanged
source = pd.read_csv(
    "45stocks_v3.csv",
    parse_dates=["DlyCalDt"]
).sort_values(
    ["PERMNO", "DlyCalDt"]
).reset_index(drop=True)

pd.testing.assert_frame_equal(
    df[original_columns],
    source[original_columns],
    check_exact=True
)

# 5. Export full dataset (including warm-up NaNs)
output_file = Path("45stocks_features40.csv")

df.to_csv(
    output_file,
    index=False,
    date_format="%Y-%m-%d"
)

# 6. Verify exported file
exported = pd.read_csv(
    output_file,
    parse_dates=["DlyCalDt"]
)

assert exported.shape == df.shape
assert exported.columns.tolist() == df.columns.tolist()
assert exported[all_features].notna().all(axis=1).sum() == 122715

pd.testing.assert_frame_equal(
    exported[original_columns],
    df[original_columns],
    check_exact=False,
    rtol=1e-12,
    atol=1e-12
)

# 7. Final summary
print("Final validation completed.")
print(f"Stocks: {df['Ticker'].nunique()}")
print(f"Total rows: {total_rows:,}")
print(f"Original columns: {len(original_columns)}")
print(f"Features: {len(all_features)}")
print(f"Total columns: {df.shape[1]}")
print(f"Warm-up rows: {warmup_rows:,}")
print(f"Complete feature rows: {valid_rows:,}")
print(f"Infinite values: {inf_counts}")
print(f"Extra missing values after warm-up: 0")
print("Original 10 columns: unchanged")
print(f"Output: {output_file}")

print("\nMissing values by feature:")
print(nan_counts.to_string())

Final validation completed.
Stocks: 45
Total rows: 124,470
Original columns: 10
Features: 40
Total columns: 50
Warm-up rows: 1,755
Complete feature rows: 122,715
Infinite values: 0
Extra missing values after warm-up: 0
Original 10 columns: unchanged
Output: 45stocks_features40.csv

Missing values by feature:
RET_1                   45
RET_3                  135
RET_5                  225
RET_10                 450
RET_20                 900
RET_39                1755
UP_RATIO_20            900
MA_DEV_10              405
MA_DEV_20              855
BREAKOUT_POS_20        855
TREND_SLOPE_20         855
TREND_T_40            1755
EFF_RATIO_20           900
GAP_1                   45
OVERNIGHT_RET_5        225
CLV_1                    0
RANGE_PCT_1             45
RV_5                   225
RV_20                  900
DOWNSIDE_RMS_20        900
ATR_PCT_14             630
PARKINSON_VOL_20       855
OVERNIGHT_VOL_20       900
MAX_DRAWDOWN_20        855
REL_DVOL_5_20          855
LOG_DVOL_20    

In [16]:
# ==================================================
# Cell 11: Target Construction (5 / 10 Days)
# ==================================================

# 1. Define prediction horizons
horizons = [5, 10]
target_cols = [f"TARGET_{h}" for h in horizons]

# 2. Verify chronological order
assert (
    df.groupby("PERMNO")["DlyCalDt"]
    .diff()
    .dropna()
    .gt(pd.Timedelta(0))
    .all()
)

# 3. Future Close-to-Close returns
# Signal: Close(t)
# Entry: t+1, subject to execution constraints
# Exit: Close(t+h)

g = df.groupby("PERMNO")["DlyClose"]

for h in horizons:
    df[f"TARGET_{h}"] = (
        g.shift(-h) / df["DlyClose"] - 1
    )

# 4. Validate target construction
assert len(target_cols) == 2
assert df.shape == (124470, 52)

values = df[target_cols].to_numpy(dtype=float)
assert np.isfinite(values[~np.isnan(values)]).all()

for h in horizons:
    col = f"TARGET_{h}"

    # Exactly h missing observations per stock
    missing = df[col].isna().groupby(df["PERMNO"]).sum()
    assert (missing == h).all()

    # Verify target formula
    expected = g.shift(-h) / df["DlyClose"] - 1

    np.testing.assert_allclose(
        df[col].to_numpy(),
        expected.to_numpy(),
        rtol=1e-12,
        atol=1e-12,
        equal_nan=True
    )

# 5. Complete ML samples
feature_mask = df[all_features].notna().all(axis=1)
target_mask = df[target_cols].notna().all(axis=1)

complete_mask = feature_mask & target_mask
complete_rows = int(complete_mask.sum())

expected_rows = 45 * (2766 - 39 - max(horizons))
assert complete_rows == expected_rows

# 6. Final validation
assert len(all_features) == 40
assert df[all_features].shape[1] == 40
assert df[["PERMNO", "DlyCalDt"]].duplicated().sum() == 0

# 7. Summary
print("Target Construction completed.")
print(f"Prediction horizons: {horizons}")
print(f"Features: {len(all_features)}")
print(f"Targets: {len(target_cols)}")
print(f"Current shape: {df.shape}")

print("\nNon-missing targets:")
print(df[target_cols].count())

print(f"\nComplete ML samples: {complete_rows:,}")
print(f"Excluded observations: {len(df) - complete_rows:,}")

print("\nExecution assumptions:")
print("Signal: Close(t)")
print("Entry: t+1, within ±5% of Close(t)")
print("Exit: Close(t+h)")
print("Entry filter and actual PnL: Backtesting stage")

display(
    df[["Ticker", "DlyCalDt", "DlyClose"] + target_cols].tail(12)
)

Target Construction completed.
Prediction horizons: [5, 10]
Features: 40
Targets: 2
Current shape: (124470, 52)

Non-missing targets:
TARGET_5     124245
TARGET_10    124020
dtype: int64

Complete ML samples: 122,265
Excluded observations: 2,205

Execution assumptions:
Signal: Close(t)
Entry: t+1, within ±5% of Close(t)
Exit: Close(t+h)
Entry filter and actual PnL: Backtesting stage


,Ticker,DlyCalDt,DlyClose,TARGET_5,TARGET_10
124458,AVGO,2025-12-15,339.81,0.004826,0.029546
124459,AVGO,2025-12-16,341.30,0.023498,0.014064
124460,AVGO,2025-12-17,326.02,0.074229,NaN
124461,AVGO,2025-12-18,329.88,0.067449,NaN
124462,AVGO,2025-12-19,340.36,0.026531,NaN
124463,AVGO,2025-12-22,341.45,0.024601,NaN
124464,AVGO,2025-12-23,349.32,-0.009218,NaN
124465,AVGO,2025-12-24,350.22,NaN,NaN
124466,AVGO,2025-12-26,352.13,NaN,NaN
124467,AVGO,2025-12-29,349.39,NaN,NaN


In [18]:
# ==================================================
# Cell 12: Data Cleaning, Time Split & CSV Export
# ==================================================

import pandas as pd
import numpy as np
from pathlib import Path

# 1. Define columns and split boundaries
target_cols = ["TARGET_5", "TARGET_10"]
required_cols = all_features + target_cols

train_end = pd.Timestamp("2021-12-31")
val_end = pd.Timestamp("2023-12-31")
test_end = pd.Timestamp("2025-12-31")

assert df.shape == (124470, 52)
assert len(all_features) == 40
assert len(target_cols) == 2
assert df["PERMNO"].nunique() == 45
assert not df.duplicated(["PERMNO", "DlyCalDt"]).any()
assert set(required_cols).issubset(df.columns)

# 2. Remove incomplete Feature / Target observations
# 39-day feature warm-up + 10-day target lookahead
data = df.dropna(subset=required_cols).copy()

removed_rows = len(df) - len(data)

assert len(data) == 122265
assert removed_rows == 2205

# 3. Chronological Train / Validation / Test split
data["Split"] = np.select(
    [
        data["DlyCalDt"] <= train_end,
        (data["DlyCalDt"] > train_end) &
        (data["DlyCalDt"] <= val_end),
        (data["DlyCalDt"] > val_end) &
        (data["DlyCalDt"] <= test_end)
    ],
    ["Train", "Validation", "Test"],
    default="Invalid"
)

assert not (data["Split"] == "Invalid").any()

# 4. Preserve all boundary observations
# No permanent purging at this stage.
# Dynamic 10-day purging will be applied
# within each Walk-Forward training fold.

data = data.reset_index(drop=True)

# 5. Final validation
assert data.shape == (122265, 53)
assert data[required_cols].notna().all().all()

assert np.isfinite(
    data[required_cols].to_numpy(dtype=float)
).all()

assert not data.duplicated(
    ["PERMNO", "DlyCalDt"]
).any()

assert data["PERMNO"].nunique() == 45
assert (data.groupby("PERMNO").size() == 2717).all()

# Verify original data and calculated values unchanged
pd.testing.assert_frame_equal(
    data.drop(columns="Split"),
    df.loc[
        df[required_cols].notna().all(axis=1)
    ].reset_index(drop=True),
    check_exact=True
)

# 6. Create separate datasets
train_df = data.loc[data["Split"] == "Train"].copy()
val_df = data.loc[data["Split"] == "Validation"].copy()
test_df = data.loc[data["Split"] == "Test"].copy()

assert len(train_df) + len(val_df) + len(test_df) == len(data)

# 7. Export final dataset
output_file = Path("data_engineered.csv")

data.to_csv(
    output_file,
    index=False,
    date_format="%Y-%m-%d",
    float_format="%.17g"
)

# 8. Verify exported CSV
exported = pd.read_csv(
    output_file,
    parse_dates=["DlyCalDt"]
)

assert exported.shape == data.shape
assert exported.columns.tolist() == data.columns.tolist()
assert exported[required_cols].notna().all().all()

# 9. Summary
print("Data Engineering completed.")
print(f"Original rows: {len(df):,}")
print(f"Removed NaN rows: {removed_rows:,}")
print(f"Final rows: {len(data):,}")
print(f"Stocks: {data['PERMNO'].nunique()}")
print(f"Features: {len(all_features)}")
print(f"Targets: {len(target_cols)}")
print(f"Total columns: {data.shape[1]}")

print("\nDataset summary:")
for name, subset in [
    ("Train", train_df),
    ("Validation", val_df),
    ("Test", test_df)
]:
    print(
        f"{name:12s} | "
        f"Rows: {len(subset):,} | "
        f"Dates: {subset['DlyCalDt'].min().date()} "
        f"to {subset['DlyCalDt'].max().date()}"
    )

print("\nValidation:")
print(f"Missing values: {data.isna().sum().sum()}")
print(f"Infinite values: {np.isinf(data[required_cols].to_numpy()).sum()}")
print("Permanent purging: Not applied")
print("Normalization: Not applied")
print(f"Output: {output_file}")

Data Engineering completed.
Original rows: 124,470
Removed NaN rows: 2,205
Final rows: 122,265
Stocks: 45
Features: 40
Targets: 2
Total columns: 53

Dataset summary:
Train        | Rows: 77,580 | Dates: 2015-03-02 to 2021-12-31
Validation   | Rows: 22,545 | Dates: 2022-01-03 to 2023-12-29
Test         | Rows: 22,140 | Dates: 2024-01-02 to 2025-12-16

Validation:
Missing values: 0
Infinite values: 0
Permanent purging: Not applied
Normalization: Not applied
Output: data_engineered.csv


In [19]:
# ==================================================
# Cell 13: Comprehensive Data Quality Validation
# ==================================================

import pandas as pd
import numpy as np
from pathlib import Path

# 1. Load engineered dataset
file = Path("data_engineered.csv")

data = pd.read_csv(
    file,
    parse_dates=["DlyCalDt"]
)

# 2. Define columns
original_cols = [
    "PERMNO", "Ticker", "DlyCalDt",
    "DlyOpen", "DlyHigh", "DlyLow",
    "DlyClose", "DlyVol", "DlyCap", "sprtrn"
]

target_cols = ["TARGET_5", "TARGET_10"]

expected_features = [
    "RET_1", "RET_3", "RET_5", "RET_10",
    "RET_20", "RET_39", "UP_RATIO_20",
    "MA_DEV_10", "MA_DEV_20", "BREAKOUT_POS_20",
    "TREND_SLOPE_20", "TREND_T_40", "EFF_RATIO_20",
    "GAP_1", "OVERNIGHT_RET_5", "CLV_1", "RANGE_PCT_1",
    "RV_5", "RV_20", "DOWNSIDE_RMS_20",
    "ATR_PCT_14", "PARKINSON_VOL_20",
    "OVERNIGHT_VOL_20", "MAX_DRAWDOWN_20",
    "REL_DVOL_5_20", "LOG_DVOL_20", "TURNOVER_20",
    "LOG_ILLIQ_20", "RET_DVOL_CORR_20",
    "SIGNED_DVOL_BAL_20",
    "SP_RET_5", "SP_VOL_20", "REL_RET_20",
    "BETA_39", "IDIO_VOL_39",
    "PEER_RET_5", "PEER_REL_RET_20",
    "PEER_VOL_20", "XS_RANK_20",
    "LOG_MCAP"
]

expected_cols = (
    original_cols + expected_features +
    target_cols + ["Split"]
)

# ==================================================
# 3. Structure validation
# ==================================================

assert data.shape == (122265, 53)
assert data.columns.tolist() == expected_cols
assert len(expected_features) == 40
assert len(set(expected_features)) == 40

assert data["PERMNO"].nunique() == 45
assert data["Ticker"].nunique() == 45

assert not data.duplicated(
    ["PERMNO", "DlyCalDt"]
).any()

# Each stock must map to exactly one ticker
assert (
    data.groupby("PERMNO")["Ticker"].nunique() == 1
).all()

assert (
    data.groupby("Ticker")["PERMNO"].nunique() == 1
).all()

# ==================================================
# 4. Date and panel consistency
# ==================================================

assert data["DlyCalDt"].min() == pd.Timestamp("2015-03-02")
assert data["DlyCalDt"].max() == pd.Timestamp("2025-12-16")

assert data["DlyCalDt"].nunique() == 2717

assert (
    data.groupby("PERMNO").size() == 2717
).all()

assert (
    data.groupby("DlyCalDt")["PERMNO"].nunique() == 45
).all()

assert data.sort_values(
    ["PERMNO", "DlyCalDt"]
).index.equals(data.index)

# ==================================================
# 5. Missing values and numerical integrity
# ==================================================

assert data.isna().sum().sum() == 0

numeric_cols = [
    c for c in data.columns
    if c not in ["Ticker", "DlyCalDt", "Split"]
]

values = data[numeric_cols].to_numpy(dtype=float)

assert np.isfinite(values).all()

# No constant features
feature_std = data[expected_features].std()

assert (feature_std > 0).all()

# ==================================================
# 6. OHLCV and market data validation
# ==================================================

prices = ["DlyOpen", "DlyHigh", "DlyLow", "DlyClose"]

assert (data[prices] > 0).all().all()
assert (data["DlyVol"] > 0).all()
assert (data["DlyCap"] > 0).all()

assert (
    data["DlyHigh"] >= data[prices].max(axis=1)
).all()

assert (
    data["DlyLow"] <= data[prices].min(axis=1)
).all()

# S&P 500 return must be identical across stocks
assert (
    data.groupby("DlyCalDt")["sprtrn"].nunique() == 1
).all()

# ==================================================
# 7. Feature range validation
# ==================================================

bounds = {
    "UP_RATIO_20": (0, 1),
    "BREAKOUT_POS_20": (0, 1),
    "EFF_RATIO_20": (0, 1),
    "CLV_1": (-1, 1),
    "MAX_DRAWDOWN_20": (0, 1),
    "RET_DVOL_CORR_20": (-1, 1),
    "SIGNED_DVOL_BAL_20": (-1, 1),
    "XS_RANK_20": (0, 1)
}

tol = 1e-10

for col, (lower, upper) in bounds.items():
    assert data[col].between(
        lower - tol, upper + tol
    ).all(), col

nonnegative = [
    "RANGE_PCT_1", "RV_5", "RV_20",
    "DOWNSIDE_RMS_20", "ATR_PCT_14",
    "PARKINSON_VOL_20", "OVERNIGHT_VOL_20",
    "MAX_DRAWDOWN_20", "SP_VOL_20",
    "IDIO_VOL_39", "PEER_VOL_20",
    "TURNOVER_20"
]

assert (data[nonnegative] >= 0).all().all()

# Independently verify market capitalization feature
np.testing.assert_allclose(
    data["LOG_MCAP"],
    np.log(data["DlyCap"] * 1000),
    rtol=1e-12,
    atol=1e-12
)

# Verify cross-sectional ranking
rank_check = (
    data.groupby("DlyCalDt")["RET_20"]
    .rank(method="average", pct=True)
)

np.testing.assert_allclose(
    data["XS_RANK_20"],
    rank_check,
    rtol=1e-12,
    atol=1e-12
)

# ==================================================
# 8. Independent return and target checks
# ==================================================

g = data.groupby("PERMNO")["DlyClose"]

# Recalculate historical returns wherever
# sufficient price history remains in this CSV.
for h in [1, 3, 5, 10, 20, 39]:
    expected = g.pct_change(
        periods=h, fill_method=None
    )
    valid = expected.notna()

    np.testing.assert_allclose(
        data.loc[valid, f"RET_{h}"],
        expected.loc[valid],
        rtol=1e-10,
        atol=1e-10
    )

# Recalculate future returns wherever
# the required future closing price is available.
for h in [5, 10]:
    expected = g.shift(-h) / data["DlyClose"] - 1
    valid = expected.notna()

    np.testing.assert_allclose(
        data.loc[valid, f"TARGET_{h}"],
        expected.loc[valid],
        rtol=1e-10,
        atol=1e-10
    )

# ==================================================
# 9. Full source consistency check
# ==================================================

reference_file = Path("45stocks_features40.csv")

if reference_file.exists():

    ref = pd.read_csv(
        reference_file,
        parse_dates=["DlyCalDt"]
    )

    assert ref.shape == (124470, 50)
    assert ref.columns.tolist() == (
        original_cols + expected_features
    )

    ref = ref.sort_values(
        ["PERMNO", "DlyCalDt"]
    ).reset_index(drop=True)

    ref_g = ref.groupby("PERMNO")["DlyClose"]

    for h in [5, 10]:
        ref[f"CHECK_TARGET_{h}"] = (
            ref_g.shift(-h) / ref["DlyClose"] - 1
        )

    keys = ["PERMNO", "DlyCalDt"]

    ref_indexed = ref.set_index(keys)
    data_indexed = data.set_index(keys)

    aligned = ref_indexed.loc[data_indexed.index]

    # Verify all original data and 40 features
    check_cols = [
        c for c in original_cols + expected_features
        if c not in keys
    ]

    pd.testing.assert_frame_equal(
        data_indexed[check_cols],
        aligned[check_cols],
        check_dtype=False,
        check_exact=False,
        rtol=1e-12,
        atol=1e-12
    )

    # Verify both targets, including final dates
    for h in [5, 10]:
        np.testing.assert_allclose(
            data_indexed[f"TARGET_{h}"],
            aligned[f"CHECK_TARGET_{h}"],
            rtol=1e-12,
            atol=1e-12
        )

    reference_status = "PASSED"

else:
    reference_status = "SKIPPED (reference file not found)"

# ==================================================
# 10. Train / Validation / Test verification
# ==================================================

split_expected = np.select(
    [
        data["DlyCalDt"] <= "2021-12-31",
        data["DlyCalDt"] <= "2023-12-31",
        data["DlyCalDt"] <= "2025-12-31"
    ],
    ["Train", "Validation", "Test"],
    default="Invalid"
)

assert np.array_equal(
    data["Split"].to_numpy(),
    split_expected
)

assert set(data["Split"]) == {
    "Train", "Validation", "Test"
}

# ==================================================
# 11. Final report
# ==================================================

print("=" * 55)
print("COMPREHENSIVE DATA VALIDATION: PASSED")
print("=" * 55)

print(f"Stocks: {data['PERMNO'].nunique()}")
print(f"Trading dates: {data['DlyCalDt'].nunique():,}")
print(f"Total rows: {len(data):,}")
print(f"Original columns: {len(original_cols)}")
print(f"Features: {len(expected_features)}")
print(f"Targets: {len(target_cols)}")
print(f"Total columns: {data.shape[1]}")

print(f"\nMissing values: {data.isna().sum().sum()}")
print(f"Infinite values: {np.isinf(values).sum()}")
print(f"Duplicate stock-date rows: "
      f"{data.duplicated(['PERMNO', 'DlyCalDt']).sum()}")

print(f"\nSource consistency: {reference_status}")
print("Feature ranges: PASSED")
print("Return and target checks: PASSED")
print("Chronological split checks: PASSED")

print("\nDataset summary:")

for split in ["Train", "Validation", "Test"]:
    subset = data[data["Split"] == split]

    print(
        f"{split:12s} | "
        f"Rows: {len(subset):,} | "
        f"Dates: {subset['DlyCalDt'].min().date()} "
        f"to {subset['DlyCalDt'].max().date()}"
    )

print("\nFeature statistics:")

display(
    data[expected_features]
    .agg(["min", "mean", "std", "max"])
    .T
    .round(6)
)

print("\nReady for normalization preparation.")
print("Dynamic purging: NOT YET APPLIED")
print("Normalization: NOT YET APPLIED")

COMPREHENSIVE DATA VALIDATION: PASSED
Stocks: 45
Trading dates: 2,717
Total rows: 122,265
Original columns: 10
Features: 40
Targets: 2
Total columns: 53

Missing values: 0
Infinite values: 0
Duplicate stock-date rows: 0

Source consistency: PASSED
Feature ranges: PASSED
Return and target checks: PASSED
Chronological split checks: PASSED

Dataset summary:
Train        | Rows: 77,580 | Dates: 2015-03-02 to 2021-12-31
Validation   | Rows: 22,545 | Dates: 2022-01-03 to 2023-12-29
Test         | Rows: 22,140 | Dates: 2024-01-02 to 2025-12-16

Feature statistics:


,min,mean,std,max
RET_1,-0.534252,0.000646,0.019901,0.522901
RET_3,-0.620103,0.001917,0.033537,0.579288
RET_5,-0.641664,0.003186,0.042768,0.766836
RET_10,-0.683131,0.006344,0.059434,0.726822
RET_20,-0.757123,0.012705,0.083855,0.882883
RET_39,-0.765034,0.024896,0.116154,1.070367
UP_RATIO_20,0.100000,0.521713,0.109055,0.950000
MA_DEV_10,-0.589405,0.002253,0.032037,0.523477
MA_DEV_20,-0.653040,0.004829,0.046573,0.607378
BREAKOUT_POS_20,0.000000,0.562063,0.302260,1.000000



Ready for normalization preparation.
Dynamic purging: NOT YET APPLIED
Normalization: NOT YET APPLIED


In [20]:
import gzip
import shutil
from pathlib import Path

source = Path("data_engineered.csv")
output = Path("data_engineered.csv.gz")

with source.open("rb") as fin, gzip.open(output, "wb", compresslevel=9) as fout:
    shutil.copyfileobj(fin, fout)

original_mb = source.stat().st_size / 1024**2
compressed_mb = output.stat().st_size / 1024**2

print(f"Original: {original_mb:.2f} MiB")
print(f"Compressed: {compressed_mb:.2f} MiB")
print(f"Reduction: {100 * (1 - compressed_mb / original_mb):.1f}%")
print(f"Browser upload compatible: {compressed_mb < 25}")

Original: 118.19 MiB
Compressed: 46.96 MiB
Reduction: 60.3%
Browser upload compatible: False
